# SEED-VII EEGNet × LoRA-LLM  —  算力平台镜像 Pipeline  (v3 MoCo)

**适用场景**: 租用 GPU 算力平台 (AutoDL / 矩池云 / 恒源云 等)，
从零拉取数据 → 预处理 → 训练，全部自动化，**无需交互**。

| 步骤 | 说明 |
|------|------|
| 1. 环境 | pip install + git clone |
| 2. 数据 | ModelScope 拉取 `DEREKVERSE/SEED-VII` (20× .mat + text_protocol.csv) |
| 3. LLM  | ModelScope 拉取 `Qwen/Qwen2.5-0.5B-Instruct` |
| 4. NPZ   | 预处理 4s 滑动窗口 → shard NPZ |
| 5. 训练  | EEGNet + LoRA-LLM + **MoCo 动量队列 (K=4096)** |
| 6. 继续  | 断点续训；验证集宏 F1 最优自动保存 best.pt |

**v3 新特性**: MoCo 动量队列将负样本从 64 → 4096；温度 warmup；LR warmup；
增强 EEG encoder (F1=16,F2=32,embed_dim=256)。

## 0. 用户配置区 (修改这里即可)

In [ ]:
import os, sys
from pathlib import Path

# ========== 必填 ==========
MODELSCOPE_TOKEN = os.environ.get('MODELSCOPE_TOKEN', '')  # 私有数据集才需要
DATASET_ID       = 'DEREKVERSE/SEED-VII'
LLM_MODEL_ID     = 'Qwen/Qwen2.5-0.5B-Instruct'

# ========== 路径 (按你的算力平台调整) ==========
WORK    = Path('/mnt/workspace')   # 持久化目录
REPO    = WORK / 'EEG_OPUS1'       # clone 到这里
MODEL_DIR    = WORK / 'models' / 'Qwen2.5-0.5B-Instruct'
DATASET_DIR  = WORK / 'seedvii_ms_dataset'
NPZ_DIR      = WORK / 'seedvii_npz'
RUN_DIR      = WORK / 'seedvii_contrastive_runs' / 'run_valence3'

# ========== 训练超参 ==========
TRAIN_BATCH_SIZE  = 96
TRAIN_EPOCHS      = 50
QUEUE_SIZE        = 4096      # MoCo 队列大小

for d in [WORK, MODEL_DIR.parent, DATASET_DIR, NPZ_DIR, RUN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('WORK:', WORK)
print('Python:', sys.version)
print('GPU:', os.popen('nvidia-smi -L 2>/dev/null || echo N/A').read().strip())

## 1. 环境安装

In [ ]:
# Clone 仓库 (如果已有则跳过)
if not (REPO / 'seedvii_modal_contrastive_lora' / 'pyproject.toml').exists():
    !rm -rf {REPO} 2>/dev/null
    !git clone https://github.com/PRIMOCOSMOS/EEG_OPUS1.git {REPO}
else:
    print('[OK] Repo already exists')

PROJ = REPO / 'seedvii_modal_contrastive_lora'

# 安装依赖
!pip install torch transformers peft modelscope h5py pyyaml tqdm scipy pandas

# 注册库
import sys
if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))

!pip install -e {PROJ} print('[OK] Environment ready')

## 2. 拉取 SEED-VII 数据集 (ModelScope)

In [ ]:
from seedvii_contrastive.scripts.download_modelscope_seedvii import find_downloaded_paths

# 检测是否已下载
EEG_ROOT, TEXT_CSV = find_downloaded_paths(DATASET_DIR)

if EEG_ROOT is None or TEXT_CSV is None:
    print('[Download] Pulling from ModelScope ...')
    !python -m seedvii_contrastive.scripts.download_modelscope_seedvii \
      --dataset-id {DATASET_ID} \
      --local-dir {DATASET_DIR} \
      --max-workers 4
    EEG_ROOT, TEXT_CSV = find_downloaded_paths(DATASET_DIR)

assert EEG_ROOT is not None, '1-20.mat not found'
assert TEXT_CSV is not None, 'text_protocol.csv not found'
print(f'[OK] EEG_ROOT={EEG_ROOT}')
print(f'[OK] TEXT_CSV={TEXT_CSV}')

## 3. 拉取 LLM (Qwen2.5-0.5B-Instruct)

In [ ]:
from modelscope import snapshot_download

if not (MODEL_DIR / 'config.json').exists():
    print('[LLM] Downloading ...')
    mp = snapshot_download(LLM_MODEL_ID, cache_dir=str(MODEL_DIR.parent))
    mp = Path(mp)
    if mp.resolve() != MODEL_DIR.resolve():
        import shutil
        if MODEL_DIR.exists(): shutil.rmtree(MODEL_DIR)
        shutil.copytree(str(mp), str(MODEL_DIR), symlinks=True)
    print(f'[LLM] Downloaded to {MODEL_DIR}')
else:
    print(f'[LLM] Already at {MODEL_DIR}')

## 4. NPZ 预处理 (4s 滑动窗口)

In [ ]:
if not (NPZ_DIR / 'index.csv').exists():
    print('[NPZ] Preprocessing ...')
    !python -m seedvii_contrastive.scripts.preprocess_npz \
      --input-root {EEG_ROOT} \
      --output-dir {NPZ_DIR} \
      --subjects 1-20 \
      --window-sec 4 --stride-sec 4 \
      --center-ratio 0.60 \
      --max-windows-per-clip 12 \
      --shard-size 512
else:
    print('[NPZ] Already preprocessed')

## 5. 写入运行时配置

In [ ]:
import yaml

base_cfg_path = PROJ / 'configs' / 'modelscope_default.yaml'
cfg = yaml.safe_load(open(base_cfg_path, 'r', encoding='utf-8'))

# 覆写路径
cfg['data'].update({
    'modelscope_dataset_id': DATASET_ID,
    'local_dataset_dir': str(DATASET_DIR),
    'eeg_root': str(EEG_ROOT),
    'text_csv_path': str(TEXT_CSV),
    'npz_dir': str(NPZ_DIR),
})
cfg['runtime']['output_dir'] = str(RUN_DIR)
cfg['model']['llm']['model_name_or_path'] = str(MODEL_DIR)
cfg['train']['batch_size'] = TRAIN_BATCH_SIZE
cfg['train']['epochs'] = TRAIN_EPOCHS
cfg['moco']['queue_size'] = QUEUE_SIZE

RUN_DIR.mkdir(parents=True, exist_ok=True)
run_cfg = RUN_DIR / 'config.yaml'
yaml.safe_dump(cfg, open(run_cfg, 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print(open(run_cfg, 'r', encoding='utf-8').read())

## 6. 训练  (MoCo v3)

In [ ]:
# 前台训练：自动检测 last.pt 续训
!python -m seedvii_contrastive.scripts.train_contrastive --config {run_cfg}

## 7. 推理 / 导出 Embedding

In [ ]:
BEST = RUN_DIR / 'best.pt'
OUT_EMB = RUN_DIR / 'val_embeddings.npz'

if BEST.exists():
    !python -m seedvii_contrastive.scripts.encode_eeg \
      --config {run_cfg} \
      --checkpoint {BEST} \
      --split val \
      --out {OUT_EMB}
    print('saved:', OUT_EMB)
else:
    print('[WARN] best.pt not found — train first')